# MAIA Demo: Kubernetes Pod

Pod-oriented version of `demo.ipynb` for running a real-neuron MAIA demo on a Kubernetes pod. Tested hardware: 2x NVIDIA A100 80 GB GPUs and 32 GB system RAM.


In [ ]:
%load_ext autoreload
%autoreload 2

### Setup Env. Vars

In [ ]:
import os

# Pod-friendly defaults. Set secrets in the shell before launching Jupyter:
#   export OPENAI_API_KEY=...
#   export HF_TOKEN=...  # if needed for FLUX access
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

for name in ["OPENAI_API_KEY", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"]:
    print(f"{name}:", "set" if os.getenv(name) else "missing")


### Imports and Params

In [ ]:
import random
import json
import torch

from maia_api import Synthetic_System, System, Tools
from utils.agents.factory import create_agent
from utils.DatasetExemplars import DatasetExemplars
from utils.ExperimentEnvironment import ExperimentEnvironment
from utils.flux import FluxDev
from utils.flux_kontext import FluxKontextDev
from utils.main_utils import *
from utils.SyntheticExemplars import SyntheticExemplars

random.seed(0000)

In [ ]:
# Layers to explore for each model
layers = {
    'resnet152': ['conv1', 'layer1', 'layer2', 'layer3', 'layer4'],
    'clip-RN50': ['layer1', 'layer2', 'layer3', 'layer4'],
    'dino_vits8': [
        'blocks.1.mlp.fc1',
        'blocks.3.mlp.fc1',
        'blocks.5.mlp.fc1',
        'blocks.7.mlp.fc1',
        'blocks.9.mlp.fc1',
        'blocks.11.mlp.fc1',
    ],
    'synthetic_neurons': ['mono', 'or', 'and'],
}

In [ ]:
agent_name: str = 'gpt-4o'  # example alternatives: 'gpt-4o-mini', or a local-* model
base_url: str = 'http://localhost:11434/v1'  # Only for local-* agents
task: str = 'neuron_description'

model: str = 'clip-RN50'
layer: str = 'layer4'
unit: str = '1673'
mode: str = "manual"

path2save: str = './results_pod_notebook'
path2prompts: str = './prompts/open/'
path2exemplars: str = './exemplars'

device: str = "0"
text2image_device: str = "cuda:0"
img2img_device: str = "cuda:0"
cpu_offload: bool = False  # keep FLUX on the GPU instead of host RAM
max_output_tokens: int = 1024
max_rounds: int = 15


### Reading Prompts, Loading NetDissect Exemplars, Building Tools and Environment

In [ ]:
# Read the API and User prompt
maia_api, user_query = return_prompt(path2prompts, setting=task)

# Load NetDissect Exemplars
unit = int(unit)
if model == 'synthetic_neurons':
    net_dissect = SyntheticExemplars(os.path.join(path2exemplars, model), path2save, layer)
    with open(os.path.join('./synthetic_neurons_dataset/labels/', f'{layer}.json')) as f:
        synthetic_neuron_data = json.load(f)
else:
    net_dissect = DatasetExemplars(path2exemplars, path2save, model, layer, [unit])

# Create directory to save results
path2save = os.path.join(path2save, agent_name, model, str(layer), str(unit))
os.makedirs(path2save, exist_ok=True)

# Setup the system to explore
if model == 'synthetic_neurons':
    gt_label = synthetic_neuron_data[unit]['label'].rsplit('_')
    print('groundtruth label:', gt_label)
    system = Synthetic_System(unit, gt_label, layer, device)
else:
    system = System(unit, layer, model, device, net_dissect.thresholds)

# Initialize tools and experiment environment
print(f'Loading FluxDev on {text2image_device}; cpu_offload={cpu_offload}')
text2image_model = FluxDev(device=text2image_device, cpu_offload=cpu_offload)
print(f'Loading FluxKontextDev on {img2img_device}; cpu_offload={cpu_offload}')
img2img_model = FluxKontextDev(device=img2img_device, cpu_offload=cpu_offload)

tools = Tools(
    path2save,
    device,
    net_dissect,
    image2text_model_name=agent_name,
    text2image_model=text2image_model,
    img2img_model=img2img_model,
)
experiment_env = ExperimentEnvironment(system, tools, globals())


### Initialize the Agent and Start the Experimentation Loop

In [ ]:
# Start the experiment log with the system prompt (maia api) and the user prompt (the query)
tools.update_experiment_log(role='system', type='text', type_content=maia_api)
tools.update_experiment_log(role='user', type='text', type_content=user_query)
ind = len(tools.experiment_log)

# Create the Agent
agent = create_agent(
    model=agent_name,
    max_attempts=5,
    max_output_tokens=max_output_tokens,
    **({'base_url': base_url} if 'local' in agent_name else {}),
)

round_count = 0
while True:
    round_count += 1

    # Ask MAIA to provide the next experiment to execute
    maia_experiment = agent.ask(tools.experiment_log)
    if maia_experiment is None:
        tools.update_experiment_log(
            role='user',
            type='text',
            type_content='Agent returned no response after retries; stopping this unit.',
        )
        tools.generate_html(path2save)
        break

    # Log MAIA's response
    tools.update_experiment_log(role='maia', type='text', type_content=str(maia_experiment))
    plot_results_notebook(tools.experiment_log[ind:])
    ind = len(tools.experiment_log)
    tools.generate_html(path2save)

    # If we exceed max_rounds, ask MAIA to finish
    if round_count > max_rounds:
        overload_instructions(tools, prompt_path=path2prompts)

    # Check for stopping condition
    if '[DESCRIPTION]' in maia_experiment:
        break

    try:
        # Execute the experiment suggested by MAIA
        output = experiment_env.execute_experiment(maia_experiment)
        if output:
            tools.update_experiment_log(role='user', type='text', type_content=output)
    except Exception as exec_e:
        tools.update_experiment_log(
            role='user',
            type='text',
            type_content=f'Error during experiment execution: {str(exec_e)}',
        )

# Save the final dialogue log as JSON plus description/label text files
save_dialogue(tools.experiment_log, path2save)
print(f'Results saved to: {path2save}')


### MAIA API

In [ ]:
print(maia_api)

### Interpretability task

In [ ]:
print(user_query)